In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import warnings

warnings.filterwarnings('ignore')

# 1. Load Data
df = pd.read_excel('CodebookData_SEMPLS_IBB.xlsx')
df = df.drop("Location", axis=1, errors='ignore')

# Note: We completely removed the renaming loop here.
# The script will natively use 'IBB1', 'IBB2', 'P1', etc.

# 2. Execute the Pipeline
df['IB_Score'] = df.filter(regex=r'^IBB').mean(axis=1)
df['Happy_Score'] = df.filter(regex=r'^H[1-4]').mean(axis=1)
df['Promo_Score'] = df.filter(regex=r'^P[1-4]').mean(axis=1)
df['Social_Score'] = df.filter(regex=r'^SI').mean(axis=1)
df['Normative_Score'] = df.filter(regex=r'^NE').mean(axis=1)
df['SC_Score'] = df.filter(regex=r'^SC').mean(axis=1)
df['Risk_SC_Penalty'] = 6 - df['SC_Score']

df['Raw_Risk'] = (1.5 * df['IB_Score']) + df['Happy_Score'] + df['Promo_Score'] + df['Social_Score'] + df['Risk_SC_Penalty']
scaler = MinMaxScaler(feature_range=(0, 100))
df['Impulse_Risk_Score'] = scaler.fit_transform(df[['Raw_Risk']])

clustering_features = df[['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'Risk_SC_Penalty']].fillna(0)
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster_ID'] = kmeans.fit_predict(clustering_features)

# Dynamically map the labels based on means to ensure accuracy across random seeds
cluster_means = df.groupby('Cluster_ID')[['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'Risk_SC_Penalty']].mean()
labels = {}
for i in range(4):
    row = cluster_means.loc[i]
    if row['Promo_Score'] > 3.5 and row['IB_Score'] > 3.5:
        labels[i] = "The Deal Chaser"
    elif row['Social_Score'] > 3.5:
        labels[i] = "The Social/Trend Shopper"
    elif row['Risk_SC_Penalty'] < 2.5: # high self control = low penalty
        labels[i] = "The Rational Spender"
    else:
        labels[i] = "The Emotional Spender"
df['Behaviour_Profile'] = df['Cluster_ID'].map(labels)

# Set style
sns.set_theme(style="whitegrid", palette="muted")

# Plot 1: Deviations (Boxplots of Questions)
# Adjusted logic to capture the clean column names dynamically
question_cols = [c for c in df.columns if c.startswith(('IBB', 'P', 'SI', 'H', 'SC', 'NE'))]
melted = df.melt(id_vars=['ID#'] if 'ID#' in df.columns else [], value_vars=question_cols, var_name='Question', value_name='Response')

# Strip numbers from the column names to get the base construct for color coding (e.g., 'IBB1' -> 'IBB')
melted['Construct'] = melted['Question'].apply(lambda x: ''.join([i for i in x if not i.isdigit()]))

plt.figure(figsize=(16, 8))
sns.boxplot(data=melted, x='Question', y='Response', hue='Construct', dodge=False)
plt.xticks(rotation=45, fontsize=10) # Adjusted rotation for cleaner reading
plt.title("Distribution & Deviations of Questionnaire Responses (1-5 Likert Scale)", fontsize=16)
plt.ylabel("Likert Score")
plt.xlabel("Survey Questions")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_deviations.png', dpi=300)
plt.close()

# Plot 2: Risk Score Distribution
plt.figure(figsize=(10, 6))
sns.histplot(data=df, x='Impulse_Risk_Score', bins=30, kde=True, color='crimson')
plt.axvline(df['Impulse_Risk_Score'].mean(), color='black', linestyle='--', label=f"Mean Risk: {df['Impulse_Risk_Score'].mean():.1f}")
plt.title("Distribution of Impulse Risk Scores", fontsize=16)
plt.xlabel("Impulse Risk Score (0-100)")
plt.ylabel("Number of Users")
plt.legend()
plt.tight_layout()
plt.savefig('plot_risk_score.png', dpi=300)
plt.close()

# Plot 3: Behaviour Profiles
plt.figure(figsize=(10, 6))
order = df['Behaviour_Profile'].value_counts().index
sns.countplot(data=df, y='Behaviour_Profile', order=order, palette='viridis')
plt.title("Count of Behavioural Profiles", fontsize=16)
plt.xlabel("Count")
plt.ylabel("Profile")
plt.tight_layout()
plt.savefig('plot_profiles.png', dpi=300)
plt.close()

# Plot 4: Cluster Visualization (PCA)
pca = PCA(n_components=2)
pca_features = pca.fit_transform(clustering_features)
df['PCA_1'] = pca_features[:, 0]
df['PCA_2'] = pca_features[:, 1]

plt.figure(figsize=(10, 8))
sns.scatterplot(data=df, x='PCA_1', y='PCA_2', hue='Behaviour_Profile', palette='viridis', s=100, alpha=0.8)
plt.title("Behavioral Clusters (2D PCA Projection)", fontsize=16)
plt.xlabel(f"Principal Component 1 ({pca.explained_variance_ratio_[0]:.1%} variance)")
plt.ylabel(f"Principal Component 2 ({pca.explained_variance_ratio_[1]:.1%} variance)")
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_clusters.png', dpi=300)
plt.close()

# Plot 5: Cluster Centers Radar/Bar
plt.figure(figsize=(12, 6))
cluster_centers = df.groupby('Behaviour_Profile')[['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'Risk_SC_Penalty']].mean().reset_index()
melted_centers = cluster_centers.melt(id_vars='Behaviour_Profile', var_name='Psychological Trigger', value_name='Average Score')

sns.barplot(data=melted_centers, x='Behaviour_Profile', y='Average Score', hue='Psychological Trigger', palette='Set2')
plt.title("Average S-O-R Scores per Behavioural Profile", fontsize=16)
plt.xlabel("Behaviour Profile")
plt.ylabel("Score")
plt.xticks(rotation=15)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.savefig('plot_cluster_centers.png', dpi=300)
plt.close()

# Save resulting subset of the final dataset
df.to_csv('analysis_results.csv', index=False)
print("Plots and results generated successfully.")

Plots and results generated successfully.


In [ ]:
import pandas as pd

df = pd.read_excel('CodebookData_SEMPLS_IBB.xlsx')
df.head()

,ID#,Gender,Location,E-Paylater User Status,Educational Background,Year of Birth,Job Status,Monthly Income,Average monthly expenditure for online shopping in relation to monthly income,IBB1: I do most of my online shopping spontaneously.,...,SC1: I am good at resisting temptation.,SC2: I never let myself lose control.,SC3: I do things that feel good at the moment but regret later.,"SC4: Sometimes I can't resist doing something, even though I know it's wrong.",SC5: I often act without considering all the alternatives.,"NE1: In my opinion, buying products or services impulsively through various online applications is WRONG.",NE2: I view the behavior of buying products or services impulsively through various online applications as IRRATIONAL.,"NE3: In my opinion, buying products or services impulsively through various online applications is not a smart choice.",NE4: I can not understand why some people buy products or services impulsively through various online applications.,"NE5: In my opinion, buying products or services impulsively through various online applications is very childish."
0,1,2,West Sumatera,1,3,1997,2,3,1,3,...,5,5,3,3,4,3,3,4,3,2
1,2,1,DKI Jakarta,2,1,2002,1,2,2,3,...,4,4,2,3,4,5,5,5,5,4
2,3,1,West Sumatera,1,1,2002,1,2,1,3,...,4,4,2,2,2,3,3,3,3,3
3,4,2,West Sumatera,2,3,2002,1,1,1,4,...,2,4,2,2,2,1,1,1,2,1
4,5,1,West Sumatera,2,3,2002,2,6,1,1,...,5,2,3,1,5,3,3,2,3,3


# Inference

In [ ]:
def process_incoming_shopper(new_data_dict, kmeans_model, scaler_model, pca_model, cluster_label_map):
    """
    Processes incoming survey responses, scores them, predicts their behavior profile,
    and issues actionable alerts.
    """
    # 1. Convert incoming dict to DataFrame
    new_df = pd.DataFrame([new_data_dict])

    # 2. Calculate Aggregates (S-O-R traits)
    ib_score = new_df.filter(regex=r'^IBB').mean(axis=1).values[0]
    happy_score = new_df.filter(regex=r'^H[1-4]').mean(axis=1).values[0]
    promo_score = new_df.filter(regex=r'^P[1-4]').mean(axis=1).values[0]
    social_score = new_df.filter(regex=r'^SI').mean(axis=1).values[0]
    sc_score = new_df.filter(regex=r'^SC').mean(axis=1).values[0]
    risk_sc_penalty = 6 - sc_score

    # 3. Calculate Risk
    raw_risk = (1.5 * ib_score) + happy_score + promo_score + social_score + risk_sc_penalty

    # Needs to be a 2D array for the scaler
    scaled_risk = scaler_model.transform([[raw_risk]])[0][0]

    # 4. Predict Cluster
    features_for_clustering = pd.DataFrame(
        [[ib_score, happy_score, promo_score, social_score, risk_sc_penalty]],
        columns=['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'Risk_SC_Penalty']
    ).fillna(0)

    cluster_id = kmeans_model.predict(features_for_clustering)[0]
    predicted_profile = cluster_label_map[cluster_id]

    # 5. Generate Alerts and Output
    print("\n" + "="*50)
    print("🔔 NEW SHOPPER INFERENCE REPORT 🔔")
    print("="*50)
    print(f"Predicted Profile : {predicted_profile}")
    print(f"Impulse Risk Score: {scaled_risk:.2f} / 100")
    print("-" * 50)

    # Custom Alert Logic
    if scaled_risk > 80:
        print("🚨 ALERT: EXTREME IMPULSE BUYER DETECTED!")
        print("   -> Action: Trigger flash sales and limited-time countdown timers.")
    elif scaled_risk < 30:
        print("🟢 ALERT: LOW RISK / RATIONAL SHOPPER.")
        print("   -> Action: Present detailed product specs, reviews, and value propositions.")

    if predicted_profile == "The Social/Trend Shopper":
        print("💡 TREND ALERT: This user responds well to social proof.")
        print("   -> Action: Show 'X people bought this in the last hour' banners.")
    elif predicted_profile == "The Emotional Spender":
        print("❤️ EMOTIONAL ALERT: Driven by mood triggers.")
        print("   -> Action: Use mood-based marketing, vibrant imagery, and lifestyle copy.")
    print("="*50 + "\n")

    return predicted_profile, scaled_risk

# ==========================================
# Example Usage of Inference
# ==========================================
# Simulating a user who answered the survey
new_shopper_survey = {
    'IBB1': 5, 'IBB2': 4, 'IBB3': 5, 'IBB4': 5, # High impulse buying
    'H1': 4, 'H2': 5, 'H3': 4, 'H4': 4,         # High happiness trigger
    'P1': 5, 'P2': 5, 'P3': 4, 'P4': 5,         # High promo response
    'SI1': 2, 'SI2': 3, 'SI3': 2, 'SI4': 2,     # Low social influence
    'SC1': 1, 'SC2': 2, 'SC3': 1, 'SC4': 2,     # Low self control (High Penalty)
    'NE1': 3, 'NE2': 3
}

# Run the incoming data through the pipeline
# Note: Ensure 'kmeans', 'scaler', 'pca', and 'labels' from the training script are in memory
profile, risk = process_incoming_shopper(new_shopper_survey, kmeans, scaler, pca, labels)


🔔 NEW SHOPPER INFERENCE REPORT 🔔
Predicted Profile : The Deal Chaser
Impulse Risk Score: 85.17 / 100
--------------------------------------------------
🚨 ALERT: EXTREME IMPULSE BUYER DETECTED!
   -> Action: Trigger flash sales and limited-time countdown timers.



## Analysis 1


In [18]:
# ---------------------------------------------------------
# 0. Imports
# ---------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler
from sklearn.decomposition import PCA
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import warnings



warnings.filterwarnings('ignore')

# Optional: ensure plots render in notebook
pio.renderers.default = "colab"   # use "browser" if running as .py script

# ---------------------------------------------------------
# 1. Load Data
# ---------------------------------------------------------
df = pd.read_excel('CodebookData_SEMPLS_IBB.xlsx')
df = df.drop("Location", axis=1, errors='ignore')

# ---------------------------------------------------------
# 2. Feature Engineering (Pipeline)
# ---------------------------------------------------------
df['IB_Score'] = df.filter(regex=r'^IBB').mean(axis=1)
df['Happy_Score'] = df.filter(regex=r'^H[1-4]').mean(axis=1)
df['Promo_Score'] = df.filter(regex=r'^P[1-4]').mean(axis=1)
df['Social_Score'] = df.filter(regex=r'^SI').mean(axis=1)
df['Normative_Score'] = df.filter(regex=r'^NE').mean(axis=1)
df['SC_Score'] = df.filter(regex=r'^SC').mean(axis=1)

df['Risk_SC_Penalty'] = 6 - df['SC_Score']

df['Raw_Risk'] = (
    (1.5 * df['IB_Score']) +
    df['Happy_Score'] +
    df['Promo_Score'] +
    df['Social_Score'] +
    df['Risk_SC_Penalty']
)

scaler = MinMaxScaler(feature_range=(0, 100))
df['Impulse_Risk_Score'] = scaler.fit_transform(df[['Raw_Risk']])

# ---------------------------------------------------------
# 3. Clustering
# ---------------------------------------------------------
clustering_features = df[
    ['IB_Score', 'Happy_Score', 'Promo_Score',
     'Social_Score', 'Risk_SC_Penalty']
].fillna(0)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster_ID'] = kmeans.fit_predict(clustering_features)

# Dynamic Cluster Labeling
cluster_means = df.groupby('Cluster_ID')[
    ['IB_Score', 'Happy_Score', 'Promo_Score',
     'Social_Score', 'Risk_SC_Penalty']
].mean()

labels = {}
for i in range(4):
    row = cluster_means.loc[i]

    if row['Promo_Score'] > 3.5 and row['IB_Score'] > 3.5:
        labels[i] = "The Deal Chaser"
    elif row['Social_Score'] > 3.5:
        labels[i] = "The Social/Trend Shopper"
    elif row['Risk_SC_Penalty'] < 2.5:
        labels[i] = "The Rational Spender"
    else:
        labels[i] = "The Emotional Spender"

df['Behaviour_Profile'] = df['Cluster_ID'].map(labels)

# ---------------------------------------------------------
# 4. PCA (Dimensionality Reduction)
# ---------------------------------------------------------
pca = PCA(n_components=2)
pca_features = pca.fit_transform(clustering_features)

df['PCA_1'] = pca_features[:, 0]
df['PCA_2'] = pca_features[:, 1]

# ---------------------------------------------------------
# 5. Interactive Plotly Visualizations
# ---------------------------------------------------------

# ---------------- Plot 1: Trait Deviations ----------------
# ==========================================
# Plot 1: Boxplots of Questions (Truncated Labels)
# ==========================================
question_cols = [c for c in df.columns if c.startswith(('IBB', 'P', 'SI', 'H', 'SC', 'NE'))]
melted = df.melt(id_vars=['ID#'] if 'ID#' in df.columns else [],
                 value_vars=question_cols,
                 var_name='Question',
                 value_name='Response')

# TRUNCATE FIX: Keep only the first 5 characters and strip extra colons/spaces
melted['Question'] = melted['Question'].astype(str).str.split(':', n=1).str[0].str.strip()

# Re-calculate the 'Construct' group for coloring (e.g., 'IBB1' -> 'IBB')
melted['Construct'] = melted['Question'].apply(lambda x: ''.join([i for i in x if not i.isdigit()]))

# Generate the plot
fig1 = px.box(melted, x='Question', y='Response', color='Construct',
              points="all",
              title="Distribution & Deviations of Specific Traits",
              labels={'Question': 'Survey Trait', 'Response': 'Likert Score (1-5)'},
              hover_data=['Question'])

# Clean up the layout
fig1.update_layout(xaxis_tickangle=-45)
fig1.show()


# ---------------- Plot 2: Risk Score Distribution ----------------
mean_risk = df['Impulse_Risk_Score'].mean()

fig2 = px.histogram(
    df,
    x='Impulse_Risk_Score',
    nbins=30,
    marginal="box",
    color_discrete_sequence=['crimson'],
    title="<b>Distribution of Impulse Risk Scores</b>",
    labels={"Impulse_Risk_Score": "Impulse Risk Score (0-100)"}
)

fig2.add_vline(
    x=mean_risk,
    line_dash="dash",
    line_color="black",
    annotation_text=f"Mean: {mean_risk:.1f}",
    annotation_position="top right"
)

fig2.show()
fig2.write_html("plot_2_risk_score.html")


# ---------------- Plot 3: Behaviour Profiles ----------------
profile_counts = df['Behaviour_Profile'].value_counts().reset_index()
profile_counts.columns = ['Behaviour_Profile', 'Count']

fig3 = px.bar(
    profile_counts,
    x='Count',
    y='Behaviour_Profile',
    orientation='h',
    color='Behaviour_Profile',
    text='Count',
    title="<b>Count of Behavioural Profiles</b>"
)

fig3.update_layout(showlegend=False)
fig3.show()
fig3.write_html("plot_3_profiles.html")


# ---------------- Plot 4: PCA Cluster Visualization ----------------
fig4 = px.scatter(
    df,
    x='PCA_1',
    y='PCA_2',
    color='Behaviour_Profile',
    size='Impulse_Risk_Score',
    hover_data=[
        'IB_Score',
        'Promo_Score',
        'Social_Score',
        'Impulse_Risk_Score'
    ],
    title=(
        f"<b>Behavioral Clusters (PCA Projection)</b><br>"
        f"<sup>PC1: {pca.explained_variance_ratio_[0]:.1%} | "
        f"PC2: {pca.explained_variance_ratio_[1]:.1%}</sup>"
    )
)

fig4.update_traces(
    marker=dict(opacity=0.8,
                line=dict(width=1, color='DarkSlateGrey'))
)

fig4.show()
fig4.write_html("plot_4_clusters.html")


# ---------------- Plot 5: Cluster Centers ----------------
cluster_centers = df.groupby('Behaviour_Profile')[
    ['IB_Score', 'Happy_Score', 'Promo_Score',
     'Social_Score', 'Risk_SC_Penalty']
].mean().reset_index()

melted_centers = cluster_centers.melt(
    id_vars='Behaviour_Profile',
    var_name='Psychological Trigger',
    value_name='Average Score'
)

fig5 = px.bar(
    melted_centers,
    x='Behaviour_Profile',
    y='Average Score',
    color='Psychological Trigger',
    barmode='group',
    title="<b>Average Psychological Triggers per Profile</b>"
)

fig5.show()
fig5.write_html("plot_5_cluster_centers.html")

# ---------------------------------------------------------
# 6. Save Final Dataset
# ---------------------------------------------------------
df.to_csv('analysis_results.csv', index=False)

print("All plots displayed and saved successfully.")
print("Results exported to 'analysis_results.csv'")

All plots displayed and saved successfully.
Results exported to 'analysis_results.csv'


# Analysis 2


In [2]:
# =============================================================================
# ENHANCED IMPULSE BUYING BEHAVIOR ANALYSIS
# =============================================================================
# This script provides comprehensive analysis including:
# - Original 5 visualizations
# - 10+ additional charts and insights
# - Statistical tests and correlations
# - Detailed summary tables
# - Profile comparisons and recommendations
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, kruskal
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
pio.renderers.default = "browser"

# =============================================================================
# 1. LOAD DATA
# =============================================================================
print("Loading data...")
df = pd.read_excel('CodebookData_SEMPLS_IBB.xlsx')
df = df.drop("Location", axis=1, errors='ignore')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

# =============================================================================
# 2. FEATURE ENGINEERING
# =============================================================================
print("\nEngineering features...")

df['IB_Score'] = df.filter(regex=r'^IBB').mean(axis=1)
df['Happy_Score'] = df.filter(regex=r'^H[1-4]').mean(axis=1)
df['Promo_Score'] = df.filter(regex=r'^P[1-4]').mean(axis=1)
df['Social_Score'] = df.filter(regex=r'^SI').mean(axis=1)
df['Normative_Score'] = df.filter(regex=r'^NE').mean(axis=1)
df['SC_Score'] = df.filter(regex=r'^SC').mean(axis=1)

df['Risk_SC_Penalty'] = 6 - df['SC_Score']

df['Raw_Risk'] = (
    (1.5 * df['IB_Score']) +
    df['Happy_Score'] +
    df['Promo_Score'] +
    df['Social_Score'] +
    df['Risk_SC_Penalty']
)

scaler = MinMaxScaler(feature_range=(0, 100))
df['Impulse_Risk_Score'] = scaler.fit_transform(df[['Raw_Risk']])

# Additional derived features
df['Total_Social_Influence'] = df['Social_Score'] + df['Normative_Score']
df['Emotional_Index'] = (df['Happy_Score'] + df['IB_Score']) / 2
df['Rational_Control'] = df['SC_Score']

# =============================================================================
# 3. CLUSTERING
# =============================================================================
print("Performing clustering...")

clustering_features = df[
    ['IB_Score', 'Happy_Score', 'Promo_Score',
     'Social_Score', 'Risk_SC_Penalty']
].fillna(0)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster_ID'] = kmeans.fit_predict(clustering_features)

# Dynamic Cluster Labeling
cluster_means = df.groupby('Cluster_ID')[
    ['IB_Score', 'Happy_Score', 'Promo_Score',
     'Social_Score', 'Risk_SC_Penalty']
].mean()

labels = {}
for i in range(4):
    row = cluster_means.loc[i]
    if row['Promo_Score'] > 3.5 and row['IB_Score'] > 3.5:
        labels[i] = "The Deal Chaser"
    elif row['Social_Score'] > 3.5:
        labels[i] = "The Social/Trend Shopper"
    elif row['Risk_SC_Penalty'] < 2.5:
        labels[i] = "The Rational Spender"
    else:
        labels[i] = "The Emotional Spender"

df['Behaviour_Profile'] = df['Cluster_ID'].map(labels)

# =============================================================================
# 4. PCA
# =============================================================================
print("Running PCA...")

pca = PCA(n_components=2)
pca_features = pca.fit_transform(clustering_features)

df['PCA_1'] = pca_features[:, 0]
df['PCA_2'] = pca_features[:, 1]

# =============================================================================
# 5. ORIGINAL VISUALIZATIONS
# =============================================================================
print("\nGenerating original visualizations...")

# Plot 1: Trait Deviations
question_cols = [c for c in df.columns if c.startswith(('IBB', 'P', 'SI', 'H', 'SC', 'NE'))]
melted = df.melt(id_vars=['ID#'] if 'ID#' in df.columns else [],
                 value_vars=question_cols,
                 var_name='Question',
                 value_name='Response')

melted['Question'] = melted['Question'].astype(str).str.split(':', n=1).str[0].str.strip()
melted['Construct'] = melted['Question'].apply(lambda x: ''.join([i for i in x if not i.isdigit()]))

fig1 = px.box(melted, x='Question', y='Response', color='Construct',
              points="all",
              title="Distribution & Deviations of Specific Traits",
              labels={'Question': 'Survey Trait', 'Response': 'Likert Score (1-5)'},
              hover_data=['Question'])
fig1.update_layout(xaxis_tickangle=-45)
fig1.write_html("plot_1_trait_deviations.html")

# Plot 2: Risk Score Distribution
mean_risk = df['Impulse_Risk_Score'].mean()
fig2 = px.histogram(
    df, x='Impulse_Risk_Score', nbins=30, marginal="box",
    color_discrete_sequence=['crimson'],
    title="<b>Distribution of Impulse Risk Scores</b>",
    labels={"Impulse_Risk_Score": "Impulse Risk Score (0-100)"}
)
fig2.add_vline(x=mean_risk, line_dash="dash", line_color="black",
               annotation_text=f"Mean: {mean_risk:.1f}",
               annotation_position="top right")
fig2.write_html("plot_2_risk_score.html")

# Plot 3: Behaviour Profiles
profile_counts = df['Behaviour_Profile'].value_counts().reset_index()
profile_counts.columns = ['Behaviour_Profile', 'Count']
fig3 = px.bar(profile_counts, x='Count', y='Behaviour_Profile', orientation='h',
              color='Behaviour_Profile', text='Count',
              title="<b>Count of Behavioural Profiles</b>")
fig3.update_layout(showlegend=False)
fig3.write_html("plot_3_profiles.html")

# Plot 4: PCA Cluster Visualization
fig4 = px.scatter(
    df, x='PCA_1', y='PCA_2', color='Behaviour_Profile',
    size='Impulse_Risk_Score',
    hover_data=['IB_Score', 'Promo_Score', 'Social_Score', 'Impulse_Risk_Score'],
    title=(f"<b>Behavioral Clusters (PCA Projection)</b><br>"
           f"<sup>PC1: {pca.explained_variance_ratio_[0]:.1%} | "
           f"PC2: {pca.explained_variance_ratio_[1]:.1%}</sup>")
)
fig4.update_traces(marker=dict(opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
fig4.write_html("plot_4_clusters.html")

# Plot 5: Cluster Centers
cluster_centers = df.groupby('Behaviour_Profile')[
    ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'Risk_SC_Penalty']
].mean().reset_index()
melted_centers = cluster_centers.melt(
    id_vars='Behaviour_Profile',
    var_name='Psychological Trigger',
    value_name='Average Score'
)
fig5 = px.bar(melted_centers, x='Behaviour_Profile', y='Average Score',
              color='Psychological Trigger', barmode='group',
              title="<b>Average Psychological Triggers per Profile</b>")
fig5.write_html("plot_5_cluster_centers.html")

print("Original visualizations complete!")

# =============================================================================
# 6. NEW ANALYSIS: CORRELATION HEATMAP
# =============================================================================
print("\n📊 Creating correlation heatmap...")

correlation_cols = ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score',
                    'Normative_Score', 'SC_Score', 'Impulse_Risk_Score']
corr_matrix = df[correlation_cols].corr()

fig6 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values.round(2),
    texttemplate='%{text}',
    textfont={"size": 10},
    colorbar=dict(title="Correlation")
))

fig6.update_layout(
    title="<b>Correlation Matrix of Psychological Factors</b>",
    xaxis_title="", yaxis_title="",
    width=700, height=700
)
fig6.write_html("plot_6_correlation_heatmap.html")

# =============================================================================
# 7. NEW ANALYSIS: RISK CATEGORIES
# =============================================================================
print("📊 Creating risk category analysis...")

# Create risk categories
df['Risk_Category'] = pd.cut(df['Impulse_Risk_Score'],
                              bins=[0, 33, 66, 100],
                              labels=['Low Risk', 'Medium Risk', 'High Risk'],
                             include_lowest=True)

risk_profile_crosstab = pd.crosstab(df['Risk_Category'], df['Behaviour_Profile'])

fig7 = go.Figure(data=[
    go.Bar(name=profile, x=risk_profile_crosstab.index,
           y=risk_profile_crosstab[profile])
    for profile in risk_profile_crosstab.columns
])

fig7.update_layout(
    title="<b>Risk Categories by Behavioural Profile</b>",
    xaxis_title="Risk Category",
    yaxis_title="Count",
    barmode='stack',
    showlegend=True
)
fig7.write_html("plot_7_risk_categories.html")

# =============================================================================
# 8. NEW ANALYSIS: RADAR CHART OF PROFILES
# =============================================================================
print("📊 Creating radar chart for profiles...")

categories = ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'SC_Score']
profile_means = df.groupby('Behaviour_Profile')[categories].mean()

fig8 = go.Figure()

for profile in profile_means.index:
    values = profile_means.loc[profile].tolist()
    values.append(values[0])  # Close the radar

    fig8.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        fill='toself',
        name=profile
    ))

fig8.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 5])),
    title="<b>Psychological Profile Comparison (Radar Chart)</b>",
    showlegend=True
)
fig8.write_html("plot_8_radar_profiles.html")

# =============================================================================
# 9. NEW ANALYSIS: DISTRIBUTION COMPARISON
# =============================================================================
print("📊 Creating distribution comparison...")

fig9 = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Impulse Buying', 'Happiness', 'Promotion',
                    'Social Influence', 'Self-Control', 'Normative')
)

scores = ['IB_Score', 'Happy_Score', 'Promo_Score',
          'Social_Score', 'SC_Score', 'Normative_Score']
positions = [(1,1), (1,2), (1,3), (2,1), (2,2), (2,3)]

for score, (row, col) in zip(scores, positions):
    for profile in df['Behaviour_Profile'].unique():
        profile_data = df[df['Behaviour_Profile'] == profile][score]
        fig9.add_trace(
            go.Violin(y=profile_data, name=profile, showlegend=(row==1 and col==1)),
            row=row, col=col
        )

fig9.update_layout(
    title_text="<b>Score Distributions Across Behavioral Profiles</b>",
    height=600,
    showlegend=True
)
fig9.write_html("plot_9_distribution_comparison.html")

# =============================================================================
# 10. NEW ANALYSIS: STATISTICAL TESTS
# =============================================================================
print("📊 Running statistical tests...")

statistical_results = []

# ANOVA tests for each score across profiles
for score in ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'SC_Score']:
    groups = [df[df['Behaviour_Profile'] == p][score].dropna()
              for p in df['Behaviour_Profile'].unique()]

    # ANOVA
    f_stat, p_value = f_oneway(*groups)

    # Kruskal-Wallis (non-parametric alternative)
    h_stat, kw_p_value = kruskal(*groups)

    statistical_results.append({
        'Factor': score,
        'F-Statistic': f_stat,
        'ANOVA p-value': p_value,
        'H-Statistic': h_stat,
        'Kruskal-Wallis p-value': kw_p_value,
        'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No'
    })

stats_df = pd.DataFrame(statistical_results)
stats_df.to_csv('statistical_tests.csv', index=False)

print("\nStatistical Test Results:")
print(stats_df.to_string())

# =============================================================================
# 11. NEW ANALYSIS: IMPULSE RISK PREDICTORS
# =============================================================================
print("\n📊 Analyzing risk score predictors...")

# Correlation with risk score
risk_correlations = df[['IB_Score', 'Happy_Score', 'Promo_Score',
                         'Social_Score', 'SC_Score', 'Normative_Score']].corrwith(
    df['Impulse_Risk_Score']
).sort_values(ascending=False)

fig10 = go.Figure(data=[
    go.Bar(x=risk_correlations.index, y=risk_correlations.values,
           marker_color=['green' if x > 0 else 'red' for x in risk_correlations.values])
])

fig10.update_layout(
    title="<b>Correlation of Factors with Impulse Risk Score</b>",
    xaxis_title="Psychological Factor",
    yaxis_title="Correlation Coefficient",
    showlegend=False
)
fig10.write_html("plot_10_risk_predictors.html")

# =============================================================================
# 12. NEW ANALYSIS: SCORE TRENDS AND OUTLIERS
# =============================================================================
print("📊 Analyzing outliers and trends...")

# Box plot with all scores
score_columns = ['IB_Score', 'Happy_Score', 'Promo_Score',
                 'Social_Score', 'SC_Score', 'Normative_Score']

melted_scores = df.melt(value_vars=score_columns,
                        var_name='Factor', value_name='Score')

fig11 = px.box(melted_scores, x='Factor', y='Score',
               color='Factor', points='outliers',
               title="<b>Score Distributions and Outliers Detection</b>")
fig11.update_layout(xaxis_tickangle=-45, showlegend=False)
fig11.write_html("plot_11_outliers.html")

# =============================================================================
# 13. NEW ANALYSIS: PROFILE CHARACTERISTICS TABLE
# =============================================================================
print("📊 Creating profile characteristics table...")

profile_summary = df.groupby('Behaviour_Profile').agg({
    'Impulse_Risk_Score': ['mean', 'std', 'min', 'max'],
    'IB_Score': 'mean',
    'Happy_Score': 'mean',
    'Promo_Score': 'mean',
    'Social_Score': 'mean',
    'SC_Score': 'mean',
    'ID#': 'count'  # or just 'count' if no ID# column
}).round(2)

profile_summary.columns = ['Risk_Mean', 'Risk_Std', 'Risk_Min', 'Risk_Max',
                           'IB_Mean', 'Happy_Mean', 'Promo_Mean',
                           'Social_Mean', 'SC_Mean', 'Count']
profile_summary.to_csv('profile_characteristics_table.csv')

print("\nProfile Characteristics:")
print(profile_summary.to_string())

# =============================================================================
# 14. NEW ANALYSIS: PERCENTILE RANKINGS
# =============================================================================
print("📊 Creating percentile rankings...")

df['Risk_Percentile'] = df['Impulse_Risk_Score'].rank(pct=True) * 100

fig12 = px.scatter(df, x=df.index, y='Risk_Percentile',
                   color='Behaviour_Profile',
                   title="<b>Individual Risk Percentile Rankings</b>",
                   labels={'y': 'Percentile Rank', 'x': 'Participant ID'})
fig12.add_hline(y=50, line_dash="dash", line_color="gray",
                annotation_text="Median")
fig12.add_hline(y=75, line_dash="dot", line_color="orange",
                annotation_text="75th Percentile")
fig12.add_hline(y=25, line_dash="dot", line_color="blue",
                annotation_text="25th Percentile")
fig12.write_html("plot_12_percentile_rankings.html")

# =============================================================================
# 15. NEW ANALYSIS: SUNBURST CHART
# =============================================================================
print("📊 Creating hierarchical sunburst chart...")

# Create hierarchical data
df_sunburst = df.copy()
df_sunburst['All'] = 'All Participants'

fig13 = px.sunburst(df_sunburst,
                    path=['All', 'Risk_Category', 'Behaviour_Profile'],
                    title="<b>Hierarchical View: Risk → Profile Distribution</b>")
fig13.write_html("plot_13_sunburst.html")

# =============================================================================
# 16. NEW ANALYSIS: PARALLEL COORDINATES
# =============================================================================
print("📊 Creating parallel coordinates plot...")

# Normalize scores for better visualization
df_parallel = df.copy()
for col in ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'SC_Score']:
    df_parallel[col + '_norm'] = (df_parallel[col] - df_parallel[col].min()) / \
                                  (df_parallel[col].max() - df_parallel[col].min())

fig14 = px.parallel_coordinates(
    df_parallel,
    dimensions=['IB_Score_norm', 'Happy_Score_norm', 'Promo_Score_norm',
                'Social_Score_norm', 'SC_Score_norm'],
    color='Impulse_Risk_Score',
    labels={
        'IB_Score_norm': 'Impulse',
        'Happy_Score_norm': 'Happiness',
        'Promo_Score_norm': 'Promotion',
        'Social_Score_norm': 'Social',
        'SC_Score_norm': 'Self-Control'
    },
    title="<b>Parallel Coordinates: Multi-dimensional Profile Analysis</b>",
    color_continuous_scale=px.colors.diverging.Tealrose
)
fig14.write_html("plot_14_parallel_coordinates.html")

# =============================================================================
# 17. NEW ANALYSIS: TIME/SEQUENCE PATTERNS (if applicable)
# =============================================================================
print("📊 Creating cumulative distribution...")

sorted_risk = df['Impulse_Risk_Score'].sort_values().reset_index(drop=True)
cumulative_pct = np.arange(1, len(sorted_risk) + 1) / len(sorted_risk) * 100

fig15 = go.Figure()
fig15.add_trace(go.Scatter(x=sorted_risk, y=cumulative_pct,
                          mode='lines', name='Cumulative %',
                          line=dict(color='royalblue', width=3)))

fig15.update_layout(
    title="<b>Cumulative Distribution of Impulse Risk Scores</b>",
    xaxis_title="Impulse Risk Score",
    yaxis_title="Cumulative Percentage",
    showlegend=False
)
fig15.add_hline(y=50, line_dash="dash", annotation_text="Median")
fig15.write_html("plot_15_cumulative_distribution.html")

# =============================================================================
# 18. NEW ANALYSIS: KEY INSIGHTS SUMMARY TABLE
# =============================================================================
print("📊 Generating key insights...")

insights = {
    'Metric': [],
    'Value': [],
    'Interpretation': []
}

# Sample size
insights['Metric'].append('Total Participants')
insights['Value'].append(str(len(df)))
insights['Interpretation'].append('Sample size for analysis')

# Risk statistics
insights['Metric'].append('Mean Risk Score')
insights['Value'].append(f"{df['Impulse_Risk_Score'].mean():.2f}")
insights['Interpretation'].append('Average impulse buying risk')

insights['Metric'].append('Risk Score Std Dev')
insights['Value'].append(f"{df['Impulse_Risk_Score'].std():.2f}")
insights['Interpretation'].append('Variability in risk levels')

# Dominant profile
dominant_profile = df['Behaviour_Profile'].value_counts().idxmax()
insights['Metric'].append('Most Common Profile')
insights['Value'].append(dominant_profile)
insights['Interpretation'].append('Largest behavioral segment')

# High risk percentage
high_risk_pct = (df['Impulse_Risk_Score'] > 66).sum() / len(df) * 100
insights['Metric'].append('High Risk %')
insights['Value'].append(f"{high_risk_pct:.1f}%")
insights['Interpretation'].append('Participants with risk > 66')

# Strongest predictor
strongest_predictor = risk_correlations.idxmax()
insights['Metric'].append('Strongest Risk Predictor')
insights['Value'].append(strongest_predictor)
insights['Interpretation'].append(f'Correlation: {risk_correlations.max():.3f}')

# Cluster separation
insights['Metric'].append('PCA Variance Explained')
insights['Value'].append(f"{(pca.explained_variance_ratio_[0] + pca.explained_variance_ratio_[1]):.1%}")
insights['Interpretation'].append('How well 2D captures variance')

insights_df = pd.DataFrame(insights)
insights_df.to_csv('key_insights_summary.csv', index=False)

print("\nKey Insights Summary:")
print(insights_df.to_string())

# =============================================================================
# 19. RECOMMENDATIONS TABLE
# =============================================================================
print("📊 Generating recommendations by profile...")

recommendations = {
    'Behaviour_Profile': [],
    'Risk_Level': [],
    'Key_Characteristics': [],
    'Intervention_Strategy': [],
    'Marketing_Approach': []
}

for profile in df['Behaviour_Profile'].unique():
    profile_data = df[df['Behaviour_Profile'] == profile]
    avg_risk = profile_data['Impulse_Risk_Score'].mean()

    recommendations['Behaviour_Profile'].append(profile)
    recommendations['Risk_Level'].append('High' if avg_risk > 66 else 'Medium' if avg_risk > 33 else 'Low')

    # Characteristics
    characteristics = []
    if profile_data['Promo_Score'].mean() > 3.5:
        characteristics.append('Promotion-sensitive')
    if profile_data['Social_Score'].mean() > 3.5:
        characteristics.append('Socially influenced')
    if profile_data['SC_Score'].mean() < 2.5:
        characteristics.append('Low self-control')
    recommendations['Key_Characteristics'].append(', '.join(characteristics) if characteristics else 'Balanced')

    # Intervention
    if avg_risk > 66:
        intervention = 'Budget apps, cooling-off periods, impulse blockers'
    elif avg_risk > 33:
        intervention = 'Spending awareness tools, goal-setting'
    else:
        intervention = 'Maintain current healthy habits'
    recommendations['Intervention_Strategy'].append(intervention)

    # Marketing
    if 'Deal Chaser' in profile:
        marketing = 'Limited-time offers, flash sales, exclusivity'
    elif 'Social' in profile:
        marketing = 'Social proof, influencer partnerships, trending items'
    elif 'Emotional' in profile:
        marketing = 'Emotional storytelling, aspirational messaging'
    else:
        marketing = 'Value propositions, quality assurance, practical benefits'
    recommendations['Marketing_Approach'].append(marketing)

recommendations_df = pd.DataFrame(recommendations)
recommendations_df.to_csv('profile_recommendations.csv', index=False)

print("\nProfile-Based Recommendations:")
print(recommendations_df.to_string())

# =============================================================================
# 20. EXPORT COMPREHENSIVE RESULTS
# =============================================================================
print("\n📊 Exporting comprehensive results...")

# Enhanced dataset
df.to_csv('analysis_results_enhanced.csv', index=False)

# Summary statistics by profile
summary_stats = df.groupby('Behaviour_Profile').agg({
    'Impulse_Risk_Score': ['count', 'mean', 'median', 'std', 'min', 'max'],
    'IB_Score': ['mean', 'std'],
    'Happy_Score': ['mean', 'std'],
    'Promo_Score': ['mean', 'std'],
    'Social_Score': ['mean', 'std'],
    'SC_Score': ['mean', 'std']
}).round(2)

summary_stats.to_csv('summary_statistics_by_profile.csv')

print("\n" + "="*70)
print("✅ ANALYSIS COMPLETE!")
print("="*70)
print(f"\n📁 Generated Files:")
print(f"   • 15 Interactive HTML Visualizations (plot_*.html)")
print(f"   • analysis_results_enhanced.csv - Full dataset with all features")
print(f"   • profile_characteristics_table.csv - Detailed profile metrics")
print(f"   • statistical_tests.csv - ANOVA and Kruskal-Wallis results")
print(f"   • key_insights_summary.csv - High-level findings")
print(f"   • profile_recommendations.csv - Actionable recommendations")
print(f"   • summary_statistics_by_profile.csv - Comprehensive stats")
print("\n" + "="*70)

# Print final summary
print("\n📊 QUICK SUMMARY:")
print(f"   Total Participants: {len(df)}")
print(f"   Behavioral Profiles Identified: {df['Behaviour_Profile'].nunique()}")
print(f"   Average Risk Score: {df['Impulse_Risk_Score'].mean():.2f}/100")
print(f"   High-Risk Individuals (>66): {(df['Impulse_Risk_Score'] > 66).sum()} ({(df['Impulse_Risk_Score'] > 66).sum()/len(df)*100:.1f}%)")
print(f"   Dominant Profile: {df['Behaviour_Profile'].value_counts().idxmax()}")
print(f"   Strongest Risk Factor: {risk_correlations.idxmax()} (r={risk_correlations.max():.3f})")
print("\n" + "="*70)

Loading data...
Dataset shape: (810, 36)
Columns: 36

Engineering features...
Performing clustering...
Running PCA...

Generating original visualizations...
Original visualizations complete!

📊 Creating correlation heatmap...
📊 Creating risk category analysis...
📊 Creating radar chart for profiles...
📊 Creating distribution comparison...
📊 Running statistical tests...

Statistical Test Results:
         Factor  F-Statistic  ANOVA p-value  H-Statistic  Kruskal-Wallis p-value Significant (α=0.05)
0      IB_Score   292.126601   3.623088e-96   343.084414            3.163487e-75                  Yes
1   Happy_Score   214.382706   2.124618e-75   240.765482            5.229256e-53                  Yes
2   Promo_Score   525.981717  5.924947e-147   439.582731            3.514170e-96                  Yes
3  Social_Score   541.548974  7.278669e-150   452.986387            4.317735e-99                  Yes
4      SC_Score    72.189352   1.438348e-29   115.237741            9.472016e-26            

## attempt 3

In [3]:
# =============================================================================
# ENHANCED IMPULSE BUYING BEHAVIOR ANALYSIS (COLAB VERSION)
# =============================================================================

import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.decomposition import PCA
from scipy import stats
from scipy.stats import chi2_contingency, f_oneway, kruskal
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import warnings
import seaborn as sns
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# ---------------------------------------------------------
# COLAB ADJUSTMENT: Set Plotly renderer to 'colab'
pio.renderers.default = "colab"
# ---------------------------------------------------------

# =============================================================================
# 1. LOAD DATA
# =============================================================================
print("Loading data...")
# Make sure 'CodebookData_SEMPLS_IBB.xlsx' is uploaded to the Colab files pane
df = pd.read_excel('CodebookData_SEMPLS_IBB.xlsx')
df = df.drop("Location", axis=1, errors='ignore')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {len(df.columns)}")

# =============================================================================
# 2. FEATURE ENGINEERING
# =============================================================================
print("\nEngineering features...")

df['IB_Score'] = df.filter(regex=r'^IBB').mean(axis=1)
df['Happy_Score'] = df.filter(regex=r'^H[1-4]').mean(axis=1)
df['Promo_Score'] = df.filter(regex=r'^P[1-4]').mean(axis=1)
df['Social_Score'] = df.filter(regex=r'^SI').mean(axis=1)
df['Normative_Score'] = df.filter(regex=r'^NE').mean(axis=1)
df['SC_Score'] = df.filter(regex=r'^SC').mean(axis=1)

df['Risk_SC_Penalty'] = 6 - df['SC_Score']

df['Raw_Risk'] = (
    (1.5 * df['IB_Score']) +
    df['Happy_Score'] +
    df['Promo_Score'] +
    df['Social_Score'] +
    df['Risk_SC_Penalty']
)

scaler = MinMaxScaler(feature_range=(0, 100))
df['Impulse_Risk_Score'] = scaler.fit_transform(df[['Raw_Risk']])

# Additional derived features
df['Total_Social_Influence'] = df['Social_Score'] + df['Normative_Score']
df['Emotional_Index'] = (df['Happy_Score'] + df['IB_Score']) / 2
df['Rational_Control'] = df['SC_Score']

# =============================================================================
# 3. CLUSTERING
# =============================================================================
print("Performing clustering...")

clustering_features = df[
    ['IB_Score', 'Happy_Score', 'Promo_Score',
     'Social_Score', 'Risk_SC_Penalty']
].fillna(0)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
df['Cluster_ID'] = kmeans.fit_predict(clustering_features)

# Dynamic Cluster Labeling
cluster_means = df.groupby('Cluster_ID')[
    ['IB_Score', 'Happy_Score', 'Promo_Score',
     'Social_Score', 'Risk_SC_Penalty']
].mean()

labels = {}
for i in range(4):
    row = cluster_means.loc[i]
    if row['Promo_Score'] > 3.5 and row['IB_Score'] > 3.5:
        labels[i] = "The Deal Chaser"
    elif row['Social_Score'] > 3.5:
        labels[i] = "The Social/Trend Shopper"
    elif row['Risk_SC_Penalty'] < 2.5:
        labels[i] = "The Rational Spender"
    else:
        labels[i] = "The Emotional Spender"

df['Behaviour_Profile'] = df['Cluster_ID'].map(labels)

# =============================================================================
# 4. PCA
# =============================================================================
print("Running PCA...")

pca = PCA(n_components=2)
pca_features = pca.fit_transform(clustering_features)

df['PCA_1'] = pca_features[:, 0]
df['PCA_2'] = pca_features[:, 1]

# =============================================================================
# 5. ORIGINAL VISUALIZATIONS
# =============================================================================
print("\nGenerating original visualizations...")

# Plot 1: Trait Deviations
question_cols = [c for c in df.columns if c.startswith(('IBB', 'P', 'SI', 'H', 'SC', 'NE'))]
melted = df.melt(id_vars=['ID#'] if 'ID#' in df.columns else [],
                 value_vars=question_cols,
                 var_name='Question',
                 value_name='Response')

melted['Question'] = melted['Question'].astype(str).str.split(':', n=1).str[0].str.strip()
melted['Construct'] = melted['Question'].apply(lambda x: ''.join([i for i in x if not i.isdigit()]))

fig1 = px.box(melted, x='Question', y='Response', color='Construct',
              points="all",
              title="Distribution & Deviations of Specific Traits",
              labels={'Question': 'Survey Trait', 'Response': 'Likert Score (1-5)'},
              hover_data=['Question'])
fig1.update_layout(xaxis_tickangle=-45)
fig1.write_html("plot_1_trait_deviations.html")
fig1.show() # COLAB ADJUSTMENT

# Plot 2: Risk Score Distribution
mean_risk = df['Impulse_Risk_Score'].mean()
fig2 = px.histogram(
    df, x='Impulse_Risk_Score', nbins=30, marginal="box",
    color_discrete_sequence=['crimson'],
    title="<b>Distribution of Impulse Risk Scores</b>",
    labels={"Impulse_Risk_Score": "Impulse Risk Score (0-100)"}
)
fig2.add_vline(x=mean_risk, line_dash="dash", line_color="black",
               annotation_text=f"Mean: {mean_risk:.1f}",
               annotation_position="top right")
fig2.write_html("plot_2_risk_score.html")
fig2.show()

# Plot 3: Behaviour Profiles
profile_counts = df['Behaviour_Profile'].value_counts().reset_index()
profile_counts.columns = ['Behaviour_Profile', 'Count']
fig3 = px.bar(profile_counts, x='Count', y='Behaviour_Profile', orientation='h',
              color='Behaviour_Profile', text='Count',
              title="<b>Count of Behavioural Profiles</b>")
fig3.update_layout(showlegend=False)
fig3.write_html("plot_3_profiles.html")
fig3.show()

# Plot 4: PCA Cluster Visualization
fig4 = px.scatter(
    df, x='PCA_1', y='PCA_2', color='Behaviour_Profile',
    size='Impulse_Risk_Score',
    hover_data=['IB_Score', 'Promo_Score', 'Social_Score', 'Impulse_Risk_Score'],
    title=(f"<b>Behavioral Clusters (PCA Projection)</b><br>"
           f"<sup>PC1: {pca.explained_variance_ratio_[0]:.1%} | "
           f"PC2: {pca.explained_variance_ratio_[1]:.1%}</sup>")
)
fig4.update_traces(marker=dict(opacity=0.8, line=dict(width=1, color='DarkSlateGrey')))
fig4.write_html("plot_4_clusters.html")
fig4.show()

# Plot 5: Cluster Centers
cluster_centers = df.groupby('Behaviour_Profile')[
    ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'Risk_SC_Penalty']
].mean().reset_index()
melted_centers = cluster_centers.melt(
    id_vars='Behaviour_Profile',
    var_name='Psychological Trigger',
    value_name='Average Score'
)
fig5 = px.bar(melted_centers, x='Behaviour_Profile', y='Average Score',
              color='Psychological Trigger', barmode='group',
              title="<b>Average Psychological Triggers per Profile</b>")
fig5.write_html("plot_5_cluster_centers.html")
fig5.show()

print("Original visualizations complete!")

# =============================================================================
# 6. NEW ANALYSIS: CORRELATION HEATMAP
# =============================================================================
print("\n📊 Creating correlation heatmap...")

correlation_cols = ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score',
                    'Normative_Score', 'SC_Score', 'Impulse_Risk_Score']
corr_matrix = df[correlation_cols].corr()

fig6 = go.Figure(data=go.Heatmap(
    z=corr_matrix.values,
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    text=corr_matrix.values.round(2),
    texttemplate='%{text}',
    textfont={"size": 10},
    colorbar=dict(title="Correlation")
))

fig6.update_layout(
    title="<b>Correlation Matrix of Psychological Factors</b>",
    xaxis_title="", yaxis_title="",
    width=700, height=700
)
fig6.write_html("plot_6_correlation_heatmap.html")
fig6.show()

# =============================================================================
# 7. NEW ANALYSIS: RISK CATEGORIES
# =============================================================================
print("📊 Creating risk category analysis...")

# Create risk categories
df['Risk_Category'] = pd.cut(df['Impulse_Risk_Score'],
                              bins=[0, 33, 66, 100],
                              labels=['Low Risk', 'Medium Risk', 'High Risk'],
                             include_lowest=True)

risk_profile_crosstab = pd.crosstab(df['Risk_Category'], df['Behaviour_Profile'])

fig7 = go.Figure(data=[
    go.Bar(name=profile, x=risk_profile_crosstab.index,
           y=risk_profile_crosstab[profile])
    for profile in risk_profile_crosstab.columns
])

fig7.update_layout(
    title="<b>Risk Categories by Behavioural Profile</b>",
    xaxis_title="Risk Category",
    yaxis_title="Count",
    barmode='stack',
    showlegend=True
)
fig7.write_html("plot_7_risk_categories.html")
fig7.show()

# =============================================================================
# 8. NEW ANALYSIS: RADAR CHART OF PROFILES
# =============================================================================
print("📊 Creating radar chart for profiles...")

categories = ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'SC_Score']
profile_means = df.groupby('Behaviour_Profile')[categories].mean()

fig8 = go.Figure()

for profile in profile_means.index:
    values = profile_means.loc[profile].tolist()
    values.append(values[0])  # Close the radar

    fig8.add_trace(go.Scatterpolar(
        r=values,
        theta=categories + [categories[0]],
        fill='toself',
        name=profile
    ))

fig8.update_layout(
    polar=dict(radialaxis=dict(visible=True, range=[0, 5])),
    title="<b>Psychological Profile Comparison (Radar Chart)</b>",
    showlegend=True
)
fig8.write_html("plot_8_radar_profiles.html")
fig8.show()

# =============================================================================
# 9. NEW ANALYSIS: DISTRIBUTION COMPARISON
# =============================================================================
print("📊 Creating distribution comparison...")

fig9 = make_subplots(
    rows=2, cols=3,
    subplot_titles=('Impulse Buying', 'Happiness', 'Promotion',
                    'Social Influence', 'Self-Control', 'Normative')
)

scores = ['IB_Score', 'Happy_Score', 'Promo_Score',
          'Social_Score', 'SC_Score', 'Normative_Score']
positions = [(1,1), (1,2), (1,3), (2,1), (2,2), (2,3)]

for score, (row, col) in zip(scores, positions):
    for profile in df['Behaviour_Profile'].unique():
        profile_data = df[df['Behaviour_Profile'] == profile][score]
        fig9.add_trace(
            go.Violin(y=profile_data, name=profile, showlegend=(row==1 and col==1)),
            row=row, col=col
        )

fig9.update_layout(
    title_text="<b>Score Distributions Across Behavioral Profiles</b>",
    height=600,
    showlegend=True
)
fig9.write_html("plot_9_distribution_comparison.html")
fig9.show()

# =============================================================================
# 10. NEW ANALYSIS: STATISTICAL TESTS
# =============================================================================
print("📊 Running statistical tests...")

statistical_results = []

for score in ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'SC_Score']:
    groups = [df[df['Behaviour_Profile'] == p][score].dropna()
              for p in df['Behaviour_Profile'].unique()]

    f_stat, p_value = f_oneway(*groups)
    h_stat, kw_p_value = kruskal(*groups)

    statistical_results.append({
        'Factor': score,
        'F-Statistic': f_stat,
        'ANOVA p-value': p_value,
        'H-Statistic': h_stat,
        'Kruskal-Wallis p-value': kw_p_value,
        'Significant (α=0.05)': 'Yes' if p_value < 0.05 else 'No'
    })

stats_df = pd.DataFrame(statistical_results)
stats_df.to_csv('statistical_tests.csv', index=False)

print("\nStatistical Test Results:")
print(stats_df.to_string())

# =============================================================================
# 11. NEW ANALYSIS: IMPULSE RISK PREDICTORS
# =============================================================================
print("\n📊 Analyzing risk score predictors...")

risk_correlations = df[['IB_Score', 'Happy_Score', 'Promo_Score',
                         'Social_Score', 'SC_Score', 'Normative_Score']].corrwith(
    df['Impulse_Risk_Score']
).sort_values(ascending=False)

fig10 = go.Figure(data=[
    go.Bar(x=risk_correlations.index, y=risk_correlations.values,
           marker_color=['green' if x > 0 else 'red' for x in risk_correlations.values])
])

fig10.update_layout(
    title="<b>Correlation of Factors with Impulse Risk Score</b>",
    xaxis_title="Psychological Factor",
    yaxis_title="Correlation Coefficient",
    showlegend=False
)
fig10.write_html("plot_10_risk_predictors.html")
fig10.show()

# =============================================================================
# 12. NEW ANALYSIS: SCORE TRENDS AND OUTLIERS
# =============================================================================
print("📊 Analyzing outliers and trends...")

score_columns = ['IB_Score', 'Happy_Score', 'Promo_Score',
                 'Social_Score', 'SC_Score', 'Normative_Score']

melted_scores = df.melt(value_vars=score_columns,
                        var_name='Factor', value_name='Score')

fig11 = px.box(melted_scores, x='Factor', y='Score',
               color='Factor', points='outliers',
               title="<b>Score Distributions and Outliers Detection</b>")
fig11.update_layout(xaxis_tickangle=-45, showlegend=False)
fig11.write_html("plot_11_outliers.html")
fig11.show()

# =============================================================================
# 13. NEW ANALYSIS: PROFILE CHARACTERISTICS TABLE
# =============================================================================
print("📊 Creating profile characteristics table...")

profile_summary = df.groupby('Behaviour_Profile').agg({
    'Impulse_Risk_Score': ['mean', 'std', 'min', 'max'],
    'IB_Score': 'mean',
    'Happy_Score': 'mean',
    'Promo_Score': 'mean',
    'Social_Score': 'mean',
    'SC_Score': 'mean',
    'ID#': 'count' if 'ID#' in df.columns else lambda x: x.count()
}).round(2)

# Handling potential missing ID# column gracefully
if 'ID#' not in df.columns:
     profile_summary.columns = ['Risk_Mean', 'Risk_Std', 'Risk_Min', 'Risk_Max',
                               'IB_Mean', 'Happy_Mean', 'Promo_Mean',
                               'Social_Mean', 'SC_Mean', 'Count']
else:
     profile_summary.columns = ['Risk_Mean', 'Risk_Std', 'Risk_Min', 'Risk_Max',
                               'IB_Mean', 'Happy_Mean', 'Promo_Mean',
                               'Social_Mean', 'SC_Mean', 'Count']

profile_summary.to_csv('profile_characteristics_table.csv')

print("\nProfile Characteristics:")
print(profile_summary.to_string())

# =============================================================================
# 14. NEW ANALYSIS: PERCENTILE RANKINGS
# =============================================================================
print("📊 Creating percentile rankings...")

df['Risk_Percentile'] = df['Impulse_Risk_Score'].rank(pct=True) * 100

fig12 = px.scatter(df, x=df.index, y='Risk_Percentile',
                   color='Behaviour_Profile',
                   title="<b>Individual Risk Percentile Rankings</b>",
                   labels={'y': 'Percentile Rank', 'x': 'Participant ID'})
fig12.add_hline(y=50, line_dash="dash", line_color="gray",
                annotation_text="Median")
fig12.add_hline(y=75, line_dash="dot", line_color="orange",
                annotation_text="75th Percentile")
fig12.add_hline(y=25, line_dash="dot", line_color="blue",
                annotation_text="25th Percentile")
fig12.write_html("plot_12_percentile_rankings.html")
fig12.show()

# =============================================================================
# 15. NEW ANALYSIS: SUNBURST CHART
# =============================================================================
print("📊 Creating hierarchical sunburst chart...")

df_sunburst = df.copy()
df_sunburst['All'] = 'All Participants'

fig13 = px.sunburst(df_sunburst,
                    path=['All', 'Risk_Category', 'Behaviour_Profile'],
                    title="<b>Hierarchical View: Risk → Profile Distribution</b>")
fig13.write_html("plot_13_sunburst.html")
fig13.show()

# =============================================================================
# 16. NEW ANALYSIS: PARALLEL COORDINATES
# =============================================================================
print("📊 Creating parallel coordinates plot...")

df_parallel = df.copy()
for col in ['IB_Score', 'Happy_Score', 'Promo_Score', 'Social_Score', 'SC_Score']:
    df_parallel[col + '_norm'] = (df_parallel[col] - df_parallel[col].min()) / \
                                  (df_parallel[col].max() - df_parallel[col].min())

fig14 = px.parallel_coordinates(
    df_parallel,
    dimensions=['IB_Score_norm', 'Happy_Score_norm', 'Promo_Score_norm',
                'Social_Score_norm', 'SC_Score_norm'],
    color='Impulse_Risk_Score',
    labels={
        'IB_Score_norm': 'Impulse',
        'Happy_Score_norm': 'Happiness',
        'Promo_Score_norm': 'Promotion',
        'Social_Score_norm': 'Social',
        'SC_Score_norm': 'Self-Control'
    },
    title="<b>Parallel Coordinates: Multi-dimensional Profile Analysis</b>",
    color_continuous_scale=px.colors.diverging.Tealrose
)
fig14.write_html("plot_14_parallel_coordinates.html")
fig14.show()

# =============================================================================
# 17. NEW ANALYSIS: TIME/SEQUENCE PATTERNS
# =============================================================================
print("📊 Creating cumulative distribution...")

sorted_risk = df['Impulse_Risk_Score'].sort_values().reset_index(drop=True)
cumulative_pct = np.arange(1, len(sorted_risk) + 1) / len(sorted_risk) * 100

fig15 = go.Figure()
fig15.add_trace(go.Scatter(x=sorted_risk, y=cumulative_pct,
                          mode='lines', name='Cumulative %',
                          line=dict(color='royalblue', width=3)))

fig15.update_layout(
    title="<b>Cumulative Distribution of Impulse Risk Scores</b>",
    xaxis_title="Impulse Risk Score",
    yaxis_title="Cumulative Percentage",
    showlegend=False
)
fig15.add_hline(y=50, line_dash="dash", annotation_text="Median")
fig15.write_html("plot_15_cumulative_distribution.html")
fig15.show()

# =============================================================================
# 18. NEW ANALYSIS: KEY INSIGHTS SUMMARY TABLE
# =============================================================================
print("📊 Generating key insights...")

insights = {
    'Metric': [],
    'Value': [],
    'Interpretation': []
}

insights['Metric'].append('Total Participants')
insights['Value'].append(str(len(df)))
insights['Interpretation'].append('Sample size for analysis')

insights['Metric'].append('Mean Risk Score')
insights['Value'].append(f"{df['Impulse_Risk_Score'].mean():.2f}")
insights['Interpretation'].append('Average impulse buying risk')

insights['Metric'].append('Risk Score Std Dev')
insights['Value'].append(f"{df['Impulse_Risk_Score'].std():.2f}")
insights['Interpretation'].append('Variability in risk levels')

dominant_profile = df['Behaviour_Profile'].value_counts().idxmax()
insights['Metric'].append('Most Common Profile')
insights['Value'].append(dominant_profile)
insights['Interpretation'].append('Largest behavioral segment')

high_risk_pct = (df['Impulse_Risk_Score'] > 66).sum() / len(df) * 100
insights['Metric'].append('High Risk %')
insights['Value'].append(f"{high_risk_pct:.1f}%")
insights['Interpretation'].append('Participants with risk > 66')

strongest_predictor = risk_correlations.idxmax()
insights['Metric'].append('Strongest Risk Predictor')
insights['Value'].append(strongest_predictor)
insights['Interpretation'].append(f'Correlation: {risk_correlations.max():.3f}')

insights['Metric'].append('PCA Variance Explained')
insights['Value'].append(f"{(pca.explained_variance_ratio_[0] + pca.explained_variance_ratio_[1]):.1%}")
insights['Interpretation'].append('How well 2D captures variance')

insights_df = pd.DataFrame(insights)
insights_df.to_csv('key_insights_summary.csv', index=False)

print("\nKey Insights Summary:")
print(insights_df.to_string())

# =============================================================================
# 19. RECOMMENDATIONS TABLE
# =============================================================================
print("📊 Generating recommendations by profile...")

recommendations = {
    'Behaviour_Profile': [],
    'Risk_Level': [],
    'Key_Characteristics': [],
    'Intervention_Strategy': [],
    'Marketing_Approach': []
}

for profile in df['Behaviour_Profile'].unique():
    profile_data = df[df['Behaviour_Profile'] == profile]
    avg_risk = profile_data['Impulse_Risk_Score'].mean()

    recommendations['Behaviour_Profile'].append(profile)
    recommendations['Risk_Level'].append('High' if avg_risk > 66 else 'Medium' if avg_risk > 33 else 'Low')

    characteristics = []
    if profile_data['Promo_Score'].mean() > 3.5:
        characteristics.append('Promotion-sensitive')
    if profile_data['Social_Score'].mean() > 3.5:
        characteristics.append('Socially influenced')
    if profile_data['SC_Score'].mean() < 2.5:
        characteristics.append('Low self-control')
    recommendations['Key_Characteristics'].append(', '.join(characteristics) if characteristics else 'Balanced')

    if avg_risk > 66:
        intervention = 'Budget apps, cooling-off periods, impulse blockers'
    elif avg_risk > 33:
        intervention = 'Spending awareness tools, goal-setting'
    else:
        intervention = 'Maintain current healthy habits'
    recommendations['Intervention_Strategy'].append(intervention)

    if 'Deal Chaser' in profile:
        marketing = 'Limited-time offers, flash sales, exclusivity'
    elif 'Social' in profile:
        marketing = 'Social proof, influencer partnerships, trending items'
    elif 'Emotional' in profile:
        marketing = 'Emotional storytelling, aspirational messaging'
    else:
        marketing = 'Value propositions, quality assurance, practical benefits'
    recommendations['Marketing_Approach'].append(marketing)

recommendations_df = pd.DataFrame(recommendations)
recommendations_df.to_csv('profile_recommendations.csv', index=False)

print("\nProfile-Based Recommendations:")
print(recommendations_df.to_string())

# =============================================================================
# 20. EXPORT COMPREHENSIVE RESULTS
# =============================================================================
print("\n📊 Exporting comprehensive results...")

df.to_csv('analysis_results_enhanced.csv', index=False)

summary_stats = df.groupby('Behaviour_Profile').agg({
    'Impulse_Risk_Score': ['count', 'mean', 'median', 'std', 'min', 'max'],
    'IB_Score': ['mean', 'std'],
    'Happy_Score': ['mean', 'std'],
    'Promo_Score': ['mean', 'std'],
    'Social_Score': ['mean', 'std'],
    'SC_Score': ['mean', 'std']
}).round(2)

summary_stats.to_csv('summary_statistics_by_profile.csv')

print("\n" + "="*70)
print("✅ ANALYSIS COMPLETE!")
print("="*70)
print(f"\n📁 Generated Files:")
print(f"   • 15 Interactive HTML Visualizations (plot_*.html)")
print(f"   • analysis_results_enhanced.csv - Full dataset with all features")
print(f"   • profile_characteristics_table.csv - Detailed profile metrics")
print(f"   • statistical_tests.csv - ANOVA and Kruskal-Wallis results")
print(f"   • key_insights_summary.csv - High-level findings")
print(f"   • profile_recommendations.csv - Actionable recommendations")
print(f"   • summary_statistics_by_profile.csv - Comprehensive stats")
print("\n" + "="*70)

print("\n📊 QUICK SUMMARY:")
print(f"   Total Participants: {len(df)}")
print(f"   Behavioral Profiles Identified: {df['Behaviour_Profile'].nunique()}")
print(f"   Average Risk Score: {df['Impulse_Risk_Score'].mean():.2f}/100")
print(f"   High-Risk Individuals (>66): {(df['Impulse_Risk_Score'] > 66).sum()} ({(df['Impulse_Risk_Score'] > 66).sum()/len(df)*100:.1f}%)")
print(f"   Dominant Profile: {df['Behaviour_Profile'].value_counts().idxmax()}")
print(f"   Strongest Risk Factor: {risk_correlations.idxmax()} (r={risk_correlations.max():.3f})")
print("\n" + "="*70)

Loading data...
Dataset shape: (810, 36)
Columns: 36

Engineering features...
Performing clustering...
Running PCA...

Generating original visualizations...


Original visualizations complete!

📊 Creating correlation heatmap...


📊 Creating risk category analysis...


📊 Creating radar chart for profiles...


📊 Creating distribution comparison...


...

📊 Creating profile characteristics table...

Profile Characteristics:
                       Risk_Mean  Risk_Std  Risk_Min  Risk_Max  IB_Mean  Happy_Mean  Promo_Mean  Social_Mean  SC_Mean  Count
Behaviour_Profile                                                                                                           
The Deal Chaser            67.19      7.44     46.57    100.00     3.51        3.82        3.63         3.52     3.02    224
The Emotional Spender      44.31      8.31     23.28     68.50     2.70        3.68        2.02         2.07     3.28    434
The Rational Spender       23.44      7.95      0.00     38.24     1.81        2.48        1.33         1.47     3.72    152
📊 Creating percentile rankings...


📊 Creating hierarchical sunburst chart...


📊 Creating parallel coordinates plot...


📊 Creating cumulative distribution...


📊 Generating key insights...

Key Insights Summary:
                     Metric                  Value                 Interpretation
0        Total Participants                    810       Sample size for analysis
1           Mean Risk Score                  46.72    Average impulse buying risk
2        Risk Score Std Dev                  16.88     Variability in risk levels
3       Most Common Profile  The Emotional Spender     Largest behavioral segment
4               High Risk %                  15.6%    Participants with risk > 66
5  Strongest Risk Predictor           Social_Score             Correlation: 0.809
6    PCA Variance Explained                  74.1%  How well 2D captures variance
📊 Generating recommendations by profile...

Profile-Based Recommendations:
       Behaviour_Profile Risk_Level                       Key_Characteristics                               Intervention_Strategy                                         Marketing_Approach
0  The Emotional Spender    

| Behaviour Profile | Risk Level | Key Characteristics | Intervention Strategy | Marketing Approach |
| :--- | :--- | :--- | :--- | :--- |
| The Deal Chaser | High | Promotion-sensitive, Socially influenced | Budget apps, cooling-off periods, impulse blockers | Limited-time offers, flash sales, exclusivity |
| The Emotional Spender | Medium | Balanced; Mood-driven | Spending awareness tools, goal-setting | Emotional storytelling, aspirational messaging |
| The Rational Spender | Low | Highly disciplined, Balanced | Maintain current healthy habits | Value propositions, quality assurance, practical benefits |
